<a href="https://colab.research.google.com/github/Keistkmiya/Tugas2-MachineLearning/blob/main/Tugas2_Chapter6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Feature Selection

## Setup and Library Imports

In [ ]:
import pandas as pd
import numpy as np
from sklearn import feature_selection

url_ev = "https://data.wa.gov/api/views/f6w7-q2d2/rows.csv?accessType=DOWNLOAD"
df_ev = pd.read_csv(url_ev)

df_ev.columns = [col.lower().replace(' ', '_') for col in df_ev.columns]

target_cols = ['model_year', 'electric_range', 'base_msrp', 'legislative_district']
numeric_cols = [col for col in target_cols if col in df_ev.columns]
df_fs = df_ev.dropna(subset=numeric_cols).copy()

print("Setup Chapter 6 Berhasil!")
print(f"Kolom yang akan kita seleksi: {numeric_cols}")

Setup Chapter 6 Berhasil!
Kolom yang akan kita seleksi: ['model_year', 'electric_range', 'legislative_district']


## Thresholding Numerical Feature Variance

Salah satu cara paling dasar dalam memilih fitur adalah dengan melihat **variansinya**. Jika sebuah kolom memiliki variansi yang sangat rendah (misalnya nilainya hampir sama semua untuk setiap baris), maka kolom tersebut tidak memberikan banyak informasi untuk membedakan antar data.

Kita menggunakan `VarianceThreshold` untuk membuang fitur yang variansinya di bawah ambang batas tertentu. Rumus variansi yang digunakan adalah:

$$\sigma^2 = \frac{\sum (x - \mu)^2}{N}$$

Jika fitur sudah distandarisasi, fitur dengan variansi rendah biasanya adalah fitur yang "statis" dan kurang berguna bagi model *Machine Learning*.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

features = df_fs[numeric_cols]
thresholder = VarianceThreshold(threshold=0.5)
features_high_variance = thresholder.fit_transform(features)

print(f"Jumlah kolom sebelum seleksi: {features.shape[1]}")
print(f"Jumlah kolom setelah seleksi variansi: {features_high_variance.shape[1]}")

selected_indices = thresholder.get_support(indices=True)
print("\nKolom yang dianggap informatif:")
print([numeric_cols[i] for i in selected_indices])

Jumlah kolom sebelum seleksi: 3
Jumlah kolom setelah seleksi variansi: 3

Kolom yang dianggap informatif:
['model_year', 'electric_range', 'legislative_district']


## Thresholding Binary Feature Variance

Fitur biner (seperti hasil One-Hot Encoding) memiliki variansi yang dihitung dengan rumus:
$$\text{Var}(x) = p(1 - p)$$
Di mana $p$ adalah proporsi kemunculan angka 1. Jika kita ingin membuang fitur yang 80% isinya adalah kategori yang sama, kita bisa mengatur ambang batas (*threshold*) variansinya.

Misalnya, jika kita ingin membuang fitur di mana salah satu kategorinya muncul lebih dari 80% kali, maka ambang batasnya adalah $0.8 \times (1 - 0.8) = 0.16$. Teknik ini sangat ampuh untuk menyederhanakan dataset yang punya terlalu banyak kolom dummy hasil encoding yang isinya kebanyakan nol.

In [ ]:
from sklearn.feature_selection import VarianceThreshold

features_binary = [[0, 1, 0],
                   [0, 1, 1],
                   [0, 1, 0],
                   [0, 1, 1],
                   [1, 0, 0]]

thresholder = VarianceThreshold(threshold=(0.8 * (1 - 0.8)))
features_selected = thresholder.fit_transform(features_binary)

print(f"Jumlah kolom sebelum seleksi: {len(features_binary[0])}")
print(f"Jumlah kolom setelah seleksi (p > 0.8): {features_selected.shape[1]}")

print("\nData hasil seleksi:")
print(features_selected)

Jumlah kolom sebelum seleksi: 3
Jumlah kolom setelah seleksi (p > 0.8): 3

Data hasil seleksi:
[[0 1 0]
 [0 1 1]
 [0 1 0]
 [0 1 1]
 [1 0 0]]


## Handling Highly Correlated Features

Dua fitur dikatakan memiliki korelasi tinggi jika informasi yang mereka berikan hampir identik. Misalnya, jika kita punya kolom 'Jarak dalam Mil' dan 'Jarak dalam Kilometer', keduanya memberikan informasi yang sama persis hanya dengan skala berbeda. Fenomena ini disebut **Multicollinearity**.

Kita menggunakan **Pearson Correlation Coefficient** ($r$) untuk mengukur hubungan ini. Nilai $r$ berkisar antara -1 hingga 1:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \sum (y_i - \bar{y})^2}}$$

Jika dua fitur memiliki korelasi di atas ambang batas (misalnya $> 0.90$), kita sebaiknya menghapus salah satunya. Hal ini membantu menyederhanakan model, mempercepat waktu pelatihan, dan membuat interpretasi model menjadi lebih mudah tanpa kehilangan informasi yang berarti.

In [ ]:
corr_matrix = df_fs[numeric_cols].corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [column for column in upper.columns if any(upper[column] > 0.9)]

print(f"Matriks Korelasi:\n{corr_matrix}")
print(f"\nFitur yang akan dihapus (Korelasi > 0.9): {to_drop}")

df_fs_reduced = df_fs.drop(columns=to_drop)

print(f"\nJumlah kolom awal: {len(numeric_cols)}")
print(f"Jumlah kolom setelah fitur redundan dihapus: {df_fs_reduced.shape[1]}")

Matriks Korelasi:
                      model_year  electric_range  legislative_district
model_year              1.000000        0.546816              0.000112
electric_range          0.546816        1.000000              0.005188
legislative_district    0.000112        0.005188              1.000000

Fitur yang akan dihapus (Korelasi > 0.9): []

Jumlah kolom awal: 3
Jumlah kolom setelah fitur redundan dihapus: 16


## Removing Irrelevant Features for Classification

Jika kita melakukan klasifikasi (misalnya menebak apakah sebuah mobil itu BEV atau PHEV), kita ingin memilih fitur yang memiliki hubungan statistik kuat dengan target tersebut. Untuk fitur numerik dan target kategorikal, kita sering menggunakan **ANOVA F-value**.

Metode ini membandingkan variansi antar grup dengan variansi di dalam grup. Fitur dengan nilai F yang tinggi berarti fitur tersebut memberikan informasi yang signifikan untuk membedakan kategori target kita. Kita menggunakan `SelectKBest` untuk memilih sejumlah $k$ fitur terbaik secara otomatis.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif

y = df_fs['electric_vehicle_type'].factorize()[0]
X = df_fs[numeric_cols]

selector = SelectKBest(score_func=f_classif, k=2)
X_best = selector.fit_transform(X, y)

print(f"Jumlah fitur awal: {X.shape[1]}")
print(f"Jumlah fitur terpilih: {X_best.shape[1]}")

mask = selector.get_support()
best_features = [col for col, selected in zip(numeric_cols, mask) if selected]

print("\n2 Fitur terbaik menurut ANOVA:")
print(best_features)

for name, score in zip(numeric_cols, selector.scores_):
    print(f"Skor {name}: {score:.2f}")

Jumlah fitur awal: 3
Jumlah fitur terpilih: 2

2 Fitur terbaik menurut ANOVA:
['model_year', 'electric_range']
Skor model_year: 6363.32
Skor electric_range: 557.69
Skor legislative_district: 489.99


## Recursive Feature Elimination (RFE)

Recursive Feature Elimination (RFE) adalah teknik *wrapper* yang bekerja dengan cara mencari subset fitur terbaik secara berulang.

**Cara kerjanya:**
1. Latih model pada semua fitur awal.
2. Hitung tingkat kepentingan setiap fitur (misalnya melalui koefisien atau *feature importance*).
3. Buang fitur yang paling tidak penting.
4. Ulangi langkah 1-3 sampai jumlah fitur yang tersisa sesuai dengan target yang kita tentukan.

Teknik ini sangat kuat karena ia mempertimbangkan interaksi antar fitur, sesuatu yang tidak bisa dilakukan oleh metode statistik sederhana seperti ANOVA.

In [ ]:
from sklearn.feature_selection import RFE
from sklearn.linear_model import LogisticRegression

estimator = LogisticRegression(max_iter=1000)
selector = RFE(estimator, n_features_to_select=2, step=1)
selector = selector.fit(X, y)

print("Hasil Seleksi RFE:")
for name, rank, support in zip(numeric_cols, selector.ranking_, selector.support_):
    status = "TERPILIH" if support else "TERELIMINASI"
    print(f"{name:20} | Ranking: {rank} | Status: {status}")

X_rfe = selector.transform(X)
print(f"\nUkuran data setelah RFE: {X_rfe.shape}")

Hasil Seleksi RFE:
model_year           | Ranking: 1 | Status: TERPILIH
electric_range       | Ranking: 2 | Status: TERELIMINASI
legislative_district | Ranking: 1 | Status: TERPILIH

Ukuran data setelah RFE: (280118, 2)
